# Veritas AI — Google Colab Training Notebook (Day 2: Dataset & Unimodal Baselines)

**Project:** Veritas AI — Multimodal Content Verification System  
**Institution:** University NLP Capstone Project  
**Target Hardware:** Google Colab GPU (T4, V100, or A100)  
**Dataset:** Fakeddit 5,000 Verified Multimodal Benchmark Pairs  

---
### Overview of Tasks in this Notebook:
1. **Environment Setup & Verification**: Check GPU, install requirements, mount Google Drive.
2. **Repository Sync**: Clone or update code from `https://github.com/yoogy-srinivash/veritas-ai.git`.
3. **Phase 1: Dataset Acquisition**: Ingest authentic Fakeddit paired multimodal data with Pillow integrity verification.
4. **Phase 2: Preprocessing & Splitting**: Stratified 70/15/15 train/val/test splits with zero leakage.
5. **Phase 3: DistilBERT Text Baseline**: Fine-tune with mixed precision (`fp16`) on GPU.
6. **Phase 4: ResNet50 Vision Baseline**: Fine-tune with mixed precision (`fp16`) on GPU.
7. **Phase 5: Evaluation & Experiment Tracking**: Test set evaluation, confusion matrices, and metrics export.
8. **Checkpoint Export**: Save all weights and JSON records directly to Google Drive for local backend deployment.

## 1. Environment Setup & GPU Verification

In [ ]:
# Check active GPU accelerator
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    print("WARNING: Running on CPU. Switch to GPU in Runtime -> Change runtime type.")
    device = torch.device('cpu')

## 2. Mount Google Drive for Persistent Storage

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/Veritas_AI_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be automatically backed up to: {CHECKPOINT_DIR}")

## 3. Clone Repository & Install Dependencies

In [ ]:
# Clone Veritas AI repository
%cd /content
!if [ ! -d "veritas-ai" ]; then git clone https://github.com/yoogy-srinivash/veritas-ai.git; fi
%cd /content/veritas-ai
!git pull origin main

# Install training dependencies
!pip install -q -r training/requirements-train.txt

## 4. Phase 1: Fakeddit Dataset Acquisition (Target: 5,000 Paired Samples)

In [ ]:
# Run Phase 1 acquisition pipeline
!python training/scripts/01_acquire_fakeddit.py 5000

## 5. Phase 2: Preprocessing & Stratified 70/15/15 Splitting

In [ ]:
# Run Phase 2 preprocessing and splitting
!python training/scripts/02_preprocess_and_split.py

## 6. Phase 3: Train DistilBERT Text Baseline (GPU)

In [ ]:
# Train DistilBERT text baseline
!python training/scripts/03_train_distilbert.py --epochs 3 --batch_size 32 --lr 2e-5

## 7. Phase 4: Train ResNet50 Image Baseline (GPU)

In [ ]:
# Train ResNet50 image baseline
!python training/scripts/04_train_resnet50.py --epochs 3 --batch_size 32 --lr 1e-4

## 8. Phase 5: Consolidate Experiments & Sync Dashboard Results

In [ ]:
# Consolidate baseline records and sync dashboard JSON
!python training/scripts/05_evaluate_and_track.py

## 9. Export Checkpoints & Artifacts to Google Drive

In [ ]:
# Copy model checkpoints and experiment records to Google Drive
!cp backend/models/distilbert_baseline.pt "$CHECKPOINT_DIR/"
!cp backend/models/resnet50_baseline.pt "$CHECKPOINT_DIR/"
!cp backend/models/experiment_results.json "$CHECKPOINT_DIR/"
!cp -r backend/models/distilbert_tokenizer/ "$CHECKPOINT_DIR/"
!cp -r experiments/ "$CHECKPOINT_DIR/"

print(f"All trained weights and experiment results safely archived in Google Drive: {CHECKPOINT_DIR}")
!ls -la "$CHECKPOINT_DIR"